<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/misc/WIP-HF_Transformers_Sentiment_Analysis_Bert_IMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HF Transformers - Sentiment Analysis - Bert/IMDB

First let's install the packages we'll be using:

In [1]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Let's retrieve the device we have available:

In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


Let's load the [IMDB dataset](https://huggingface.co/datasets/stanfordnlp/imdb), which pairs IMDB reviews with a binary sentiment classification (positive, negative).

In [3]:
from datasets import load_dataset

raw_dataset = load_dataset("stanfordnlp/imdb")
raw_dataset

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

The dataset has a `train` split with 25k samples, and a `test` split with 25k samples as well. There is also an extra `unsupervised` split with 50k entries, it's not obvious what that split is for, let's inspect it:

In [4]:
raw_dataset["unsupervised"][0]

{'text': 'This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything else, maybe except the movie "Leon" wonderfully played by Jean Reno and Natalie Portman. But... What can I say about this one? This is the best movie Anne Parillaud has ever played in (See please "Frankie Starlight", she\'s speaking English there) to see what I mean. The story of young punk girl Nikita, taken into the depraved world of the secret government forces has been exceptionally over used by Americans. Never mind the "Point of no return" and especially the "La femme Nikita" TV series. They cannot compare the original believe me! Trash these videos. Buy this one, do not rent it, BUY it. BTW beware of the subtitles of the LA company which "translate" the US release. What a disgrace! If you cant understand French, get a dubbed version. But you\'ll regret later :)',
 'label': -1}

In [5]:
list(set([x["label"] for x in raw_dataset["unsupervised"]]))

[-1]

We've confirmed that `unsupervised` is what the name says, an unsupervised split that doesn't have any labels. It won't be useful for the sentiment analysis task, but could be useful for generating similar reviews for example (text generation task).

Let's learn more about the dataset features:

In [6]:
raw_dataset["train"].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

Let's inspect some samples from the `train` split:

In [7]:
raw_dataset["train"][:5]["text"]

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

Let's check the variability of the text lengths:

In [8]:
raw_dataset = raw_dataset.map(lambda x: {"text_length" : len(x["text"].split())})
raw_dataset

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 50000
    })
})

In [9]:
min(raw_dataset["train"]["text_length"]), max(raw_dataset["train"]["text_length"]), sum(raw_dataset["train"]["text_length"]) / len(raw_dataset["train"]["text_length"])

(10, 2470, 233.7872)

Let's load a model to fine-tune:

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


One thing, I'm not sure about is how the uncased model deals with casing. Can it not tokenized cased sequences? Or does it lowercase the input first? Let's check:

In [11]:
tokenizer.convert_ids_to_tokens(tokenizer("This is a Test!")["input_ids"])

['[CLS]', 'this', 'is', 'a', 'test', '!', '[SEP]']

The model lowercases before tokenizing, which means we can feed it the dataset without lowercasing the text first. Let's now tokenize the dataset:

In [12]:
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {num_cores}")

def _tokenize(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = raw_dataset.map(
    _tokenize,
    batched=True, # Enables batch processing
    num_proc=num_cores, # Adjust this based on your CPU cores
    remove_columns=["text", "text_length"] # Optional: reduces memory usage
)
tokenized_dataset

Number of CPU cores: 12


Map (num_proc=12):   0%|          | 0/25000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/25000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

Let's create a function to evaluate the model. This function will run through the test set and calculate the average prediction accuracy on it:

The starting accuracy is ~50%. This means that by default this model is pretty much random at being able to do sentiment classification. Let's setup a training run to fine-tune the model on this dataset:

In [15]:
import numpy as np
import evaluate
from transformers import Trainer
from transformers import TrainingArguments

# TODO: try to grok (use AdamW, adapt learning rate)
def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred  # eval_pred is a tuple (logits, labels)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="imdb_bert_fine_tuned",  # Output directory
    resume_from_checkpoint=True,  # This is the key line!
    num_train_epochs=1,  # Adjust as needed
    per_device_train_batch_size=32, # Start with a large batch size
    per_device_eval_batch_size=32, # Start with a large batch size
    gradient_accumulation_steps=1,  # Adjust if you run out of memory
    fp16=False, # Try bf16 first!
    bf16=True, # Brain Floating Point for A100 - try this first
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=1e-7,  # Adjust as needed
    weight_decay=0.01,  # Adjust as needed
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    warmup_steps=0,
    logging_steps=50, # Log every 50 steps,
    eval_steps=5 * len(tokenized_dataset["train"]) // 64#training_args.per_device_train_batch_size # Evaluate every 5 epochs
)

trainer = Trainer(
    model, # the instantiated 🤗 Transformers model to be trained
    training_args, # training arguments, defined above
    train_dataset=tokenized_dataset["train"], # The dataset to train the model on
    eval_dataset=tokenized_dataset["test"], # The dataset to evaluate the model on
    data_collator=data_collator, # defaults to DataCollatorWithPadding if not provided
    tokenizer=tokenizer, # The tokenizer to be used
    compute_metrics=compute_metrics  # Add the compute_metrics function
)
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-15-98acb0e4d11c>:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.692800,0.686045,0.533920


TrainOutput(global_step=782, training_loss=0.6935183629965234, metrics={'train_runtime': 184.9861, 'train_samples_per_second': 135.145, 'train_steps_per_second': 4.227, 'total_flos': 6575737273320960.0, 'train_loss': 0.6935183629965234, 'epoch': 1.0})

Now that the model is trained, let's run the evaluation again:

In [16]:
tokenized = tokenizer([
    "I have *had* it with these motherfucking *snakes* on this motherfucking *plane*!",
    "Spam, Spam, Spam, Spam! Spam, Spam, Spam, Spam! Lovely Spam, wonderful Spam!"
], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
output = model(**tokenized)
output.logits.shape, output.logits

(torch.Size([2, 2]),
 tensor([[-0.3145, -0.2441],
         [ 0.0107, -0.1943]], device='cuda:0', grad_fn=<ToCopyBackward0>))

In [17]:
predictions = torch.argmax(output.logits, dim=1).cpu().numpy()
predictions

array([1, 0])

In [18]:
LABELS = raw_dataset["train"].features["label"].names
[LABELS[x] for x in predictions]

['pos', 'neg']

In [22]:
predictions = trainer.predict(tokenized_dataset["test"])
predictions.predictions

array([[ 0.18261719, -0.03076172],
       [ 0.20703125, -0.05053711],
       [ 0.03686523, -0.11865234],
       ...,
       [ 0.12988281,  0.00866699],
       [ 0.25      ,  0.02893066],
       [ 0.04858398, -0.06640625]], dtype=float32)

In [28]:
predicted_labels = np.argmax(predictions.predictions, axis=1)
predicted_labels

array([0, 0, 0, ..., 0, 0, 0])

In [26]:
true_labels = predictions.label_ids
true_labels

array([0, 0, 0, ..., 1, 1, 1])

In [30]:
misclassified_indices = np.where(predicted_labels != true_labels)[0]
misclassified_indices

array([   36,   120,   121, ..., 24997, 24998, 24999])

In [63]:
misclassified_sample = tokenized_dataset["test"][int(misclassified_indices[0])]
misclassified_true_label = misclassified_sample["label"]
misclassified_true_label

0

In [79]:
from transformers import AutoConfig

model_name = "bert-base-uncased"  # Replace with your model
config = AutoConfig.from_pretrained(model_name)
print(config.max_position_embeddings)  # Max tokens the model can handle

512


In [94]:
decoded_text = tokenizer.decode(misclassified_sample["input_ids"], skip_special_tokens=True)
print(decoded_text)

beware, my lovely ( 1952 ) dir : harry horner < br / > < br / > production : the filmmakers / rko radio pictures < br / > < br / > credulity - straining thriller from the pioneering producer team of collier young and ida lupino, aka the filmmakers ( with lupino pitching in with some uncredited direction ). < br / > < br / > robert ryan is the ' peril ' and ida lupino is the ' woman ' in this entry in the ' woman in peril ' style film. ryan plays howard wilton, a tightly - wound psychotic handyman drifter ( noooo, ryan? i know, hard to believe ). lupino is the lonely war widow, helen gordon, who hires howard to do some work around her house. things go downhill from there as howard makes helen a prisoner in her own home. < br / > < br / > howard has a nasty secret, not that he could reveal it. you see, consciousness is a real challenge for him. maintaining it, that is. he has an unfortunate habit of coming to and finding his employers dead. this is part of the film ' s problem. the natur

In [91]:
result = model(input_ids=torch.tensor([misclassified_sample["input_ids"]]).to(DEVICE), attention_mask=torch.tensor([misclassified_sample["attention_mask"]]).to(DEVICE))
result

SequenceClassifierOutput(loss={'logits': tensor([[0.1299, 0.1787]], device='cuda:0', grad_fn=<ToCopyBackward0>)}, logits=tensor([[0.1299, 0.1787]], device='cuda:0', grad_fn=<ToCopyBackward0>), hidden_states=None, attentions=None)

In [92]:
result = torch.argmax(result.logits, dim=1).cpu().numpy()[0]
LABELS[result]

'pos'

NOTE: chatgpt can classify this sentence correctly
TODO: extract validation split and run eval on test split